# Test Case for Marketing Analytics

# Information
- **Author:** Leila Gharavi
- **Email:** `gharavi95@gmail.com`
- **Date:** 18 Nov 2026

#### Relevant links
- [Google Merchandize Store](https://merch.google/$0)
- BigQuery dataset: `bigquery-public-data:ga4
obfuscated_sample_ecommerce`
- GCP project: `crystalloids-analytics`
  - Dataset `analytics` created with US location (due to raw source origin)
  - Dataset `ml` created with US location (due to raw source origin)
- [GitHub repo](https://github.com/leilagharavi/analytics-test-case.git)


## Timeline
- 10:00 - 10:30 Familirized with hardware setup
- 10:30 - 11:00 Setting up notebook skeleton, GCP project, and gitHub repository
- 11:00 - 11:30 Install dbt and set up dbt project, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=hsmD51nKIOni&line=6&uniqifier=1)
- 11:30 - 12:00 swithc to local dbt development and set up BQ connection with SA
- 12:00 - 13:00 set up dbt project and data models, [see here](https://colab.research.google.com/github/leilagharavi/analytics-test-case/blob/main/walkthrough.ipynb#scrollTo=mR2Ds6hhPw--&line=26&uniqifier=1)

## Process

### dbt

#### Installation





In [ ]:
# Uninstall any conflicting packages if present
!pip uninstall -y dbt-bigquery sqlglot dbt-state

# Install dbt-bigquery cleanly with user permissions
!pip install --user dbt-bigquery

## Check dbt version
!dbt --version

#### dbt project set up
- I choose to keep the notebook file in the same repo where the dbt project lives.

In [ ]:
!dbt init marketing_analytics

### Local dbt setup

- Service Account: `marketing_analytics_sa`
- Permissions: `BigQuery Admin` and `BigQuery Job User`
- JSON key downloaded on local (independant) path
```Bash
dbt init marketing_analytics
dbt debug
```
### Create source (raw layer)
```Bash
touch marketing_analytics/seeds/sources.yml
```
Which contains the following code:
```YAML
version: 2

sources:
  - name: ga4_raw
    database: bigquery-public-data
    schema: ga4_obfuscated_sample_ecommerce
    tables:
      - name: events
        identifier: events_*
```
### Create layers

#### Staging layer
- The data is cleaned and pre-processed before implementing business logic in the next layers.
- Materialized as view to limit computational usage.
```Bash
mkdir marketing_analytics/models/staging/
touch marketing_analytics/models/staging/stg_ga4_events.sql
```
The dbt tests for this model is created by running
```Bash
touch marketing_analytics/models/staging/schema.yml
```
which includes the following code:
```YAML
version: 2

models:
  - name: stg_ga4_events
    description: "Cleaned and flattened GA4 event-level staging model."
    columns:
      - name: user_pseudo_id
        description: "Unique identifier for the user."
        tests:
          - not_null
          
      - name: event_timestamp
        description: "Exact timestamp of the event."
        tests:
          - not_null

      - name: event_name
        description: "The name of the GA4 event logged."
        tests:
          - not_null
```

#### Marts layer
- Materialized as table and includes features for ML model training.
```Bash
mkdir marketing_analytics/models/marts/
touch marketing_analytics/models/marts/user_features.sql
touch marketing_analytics/models/marts/schema.yml
```
The model is created as
```SQL
SELECT
    user_pseudo_id,
    
    -- Behavioral Aggregations
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(1) AS total_events,
    COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN session_id END) AS purchase_session_count,
    COALESCE(SUM(event_value), 0) AS total_spend_usd,
    
    -- Contextual Attributes (Taking the first/any value for user demographics)
    ANY_VALUE(device_category) AS device_category,
    ANY_VALUE(country) AS country,
    
    -- Machine Learning Target Label: Did the user make a purchase? (1 or 0)
    IF(COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN session_id END) > 0, 1, 0) AS label_made_purchase

FROM {{ ref('stg_ga4_events') }}
GROUP BY user_pseudo_id
```
Moreover, a schema file is created to include tests and descriptions as follows:

```YAML
version: 2

models:
  - name: user_features
    description: "User-level aggregated feature table built from GA4 staging events, optimized for BQML and user behavior analytics."
    columns:
      - name: user_pseudo_id
        description: "Primary key: Unique pseudo identifier for each user."
        tests:
          - unique
          - not_null
          
      - name: total_sessions
        description: "Total distinct sessions initiated by the user."
        tests:
          - not_null
          
      - name: label_made_purchase
        description: "Binary target label for BQML: 1 if the user purchased, 0 otherwise."
        tests:
          - not_null
          - accepted_values:
              values: [0, 1]
              quote: false
```
Note that the `quote: false` in the end was added to avoid errors due to string mishandling and type mismatch.

### Create analysis placeholders
The following file is created to later include the query for creating the ML model.

```Bash
touch marketing_analytics/analyses/ml_model_training.sql
```





## Key Decisions

### AI usage declaration
- Solely used Google Gemini due to time-bound test-case nature of the project.
- No text was copied into console/prompt box from the assignment doc due to security concerns.
- There was an error during installation of dbt due to incompatible dependancies, for which I copied the error codes in my prompts to help resolve them.

### Process reflections
- The following were new experience, which no always worked well regardless of getting effective help from AI and time-bound setting of expectations for myself:
  - setting up Colab/GitHub connection
  - using notebook as terminal was not a straightforward experience.
- I decided to skip creating intermediate models due to limited time.

### Decisions on IaC implementation


### Decisions on ML platform of choice

